# 70장 · 문서 상태와 전체 답변을 같은 DB에 저장하기

파일 이름은 보이는데 그 문서로 질문하면 근거가 없습니다. 문서를 접수한 때, 원문을 읽어 검색할 수 있게 된 때, 답변을 대화 이력에 저장한 때는 서로 다릅니다. 작은 SQL 데이터베이스에서 각 단계의 행을 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 접수한 문서를 검색 가능 상태로 바꾸기

아래는 70장의 소유자·문서·상태 셋을 남긴 SQLite 실습입니다. 실제 PostgreSQL 버전은 UUID와 복합 외래키로 문서·대화·출처를 연결합니다. 여기서는 업로드 본문 한 건의 변화를 관찰합니다.



**볼 곳**　‘파일 전송’에서 출발해 ‘수신 상태’를 거친 뒤 ‘후속 처리’가 되는 순서를 따라가세요.

**애니메이션의 의미**　파일 수신 상태를 남긴 뒤 후속 처리를 이어갑니다.


### 셀 01 · 파일을 접수하고 상태 읽기

과제 안내를 alice 문서로 접수합니다. 텍스트를 저장하기 전에는 ready로 표시하지 않습니다.

In [1]:
import sqlite3, json
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE docs(id TEXT PRIMARY KEY, owner TEXT, status TEXT, body TEXT)")
db.execute("CREATE TABLE turns(conversation TEXT, owner TEXT, result TEXT)")
db.execute("INSERT INTO docs VALUES (?, ?, 'received', NULL)", ("doc-1", "alice"))
print(db.execute("SELECT status FROM docs WHERE id=? AND owner=?", ("doc-1", "alice")).fetchone()[0])

received


received는 요청을 접수했다는 뜻입니다. 원문 처리나 출처 준비 성공을 보장하지 않습니다.

### 셀 02 · UTF-8 원문을 읽고 상태 전환

기존 상태가 received일 때만 한 번 전환합니다. 자료가 준비된 뒤에만 검색 후보에 넣으세요.

같은 셀을 두 번 실행해 변경 행 수를 비교하세요. 처음에는 received 상태의 문서를 찾아 바꾸고, 두 번째에는 이미 ready라서 UPDATE에 맞는 행이 없습니다. 조회한 최종 상태는 둘 다 ready입니다.

In [2]:
raw = "과제는 학습 관리 페이지의 과제함에 제출합니다.".encode("utf-8")
with db:
    change = db.execute("UPDATE docs SET body=?, status='ready' WHERE id=? AND owner=? AND status='received'",
               (raw.decode("utf-8"), "doc-1", "alice"))
print("변경한 행:", change.rowcount)
print(db.execute("SELECT status, body FROM docs WHERE id='doc-1'").fetchone())

변경한 행: 1
('ready', '과제는 학습 관리 페이지의 과제함에 제출합니다.')


ready와 읽은 문장이 함께 돌아옵니다. 읽을 수 없는 바이트를 넣었다면 ready로 바꾸는 줄까지 진행하면 안 됩니다.

## 결과와 출처를 같은 턴에 저장하기

답변 문장만 저장하면 같은 대화를 다시 열 때 어떤 문서와 쪽수를 썼는지 복원하기 어렵습니다. 서비스 결과 A1을 통째로 저장하고 다시 읽습니다. 수업용 doc-1 식별자는 업로드한 문서를 가리킵니다.



**볼 곳**　‘서비스 응답’를 ‘출처와 함께 저장’에 남기는 시점과 ‘다시 연 대화’를 확인하는 시점을 구분하세요.

**애니메이션의 의미**　답과 출처를 함께 저장해야 대화를 다시 열 수 있습니다.


### 셀 03 · 저장한 답변에서 출처 복원

소유자의 준비된 문서에서 원문을 읽고, 답변과 함께 출처·호출 수를 저장합니다.

In [3]:
body = db.execute("SELECT body FROM docs WHERE owner=? AND id=? AND status='ready'",
                  ("alice", "doc-1")).fetchone()[0]
source = {"id": "doc-1:p1", "page": 1, "source": "제출 안내", "text": body}
a1 = {"status": "answered", "answer": body + " [doc-1:p1]", "sources": [source], "generation_calls": 1}
with db:
    db.execute("INSERT INTO turns VALUES (?,?,?)", ("chat-a", "alice", json.dumps(a1, ensure_ascii=False)))
saved = db.execute("SELECT result FROM turns WHERE owner=? AND conversation=?", ("alice", "chat-a")).fetchone()
print(json.loads(saved[0])["sources"][0]["text"])

과제는 학습 관리 페이지의 과제함에 제출합니다.


다시 조회한 행에도 원문이 남습니다. 이 코드는 저장까지 성공했습니다. 답변 생성 뒤 저장에 실패했다면 응답이 생성된 사실과 이력이 남았는지를 별도로 표시해야 합니다.

### 과제 04 · 같은 대화 이름을 다른 사용자로 조회

bob이 chat-a를 조회할 때 행이 보이지 않는 조건을 완성하세요.

In [4]:
# 두 자리표시자에 소유자와 대화 이름을 넣으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>row = db.execute(&#x27;SELECT result FROM turns WHERE owner=? AND conversation=?&#x27;, (&#x27;bob&#x27;, &#x27;chat-a&#x27;)).fetchone()
print(row)</code></pre>
<p>실행 결과</p><pre><code>None</code></pre>
<p>None입니다. 이 셀은 SQL 조회 조건을 확인했습니다. 실제 배포용 로그인과 DB 권한 설정은 별도로 검증해야 합니다.</p>
</details>

## 돌아보기

파일 접수 후 원문을 읽지 못한 경우와 답변을 만든 뒤 턴 저장이 실패한 경우에 각각 어떤 기록을 확인할지 말해 보세요.